# Reefugia: Analysis workflow
End-to-end single-year coral bleaching analysis using Copernicus Marine Service
data. Follows the same steps as the `reefugia` CLI (without saving outputs); reuses the zarr caches in `data/`.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import cartopy.feature as cfeature

from reefugia import fetch, detect, analyse, visualise
from reefugia.fetch import REGIONS
from reefugia.constants import KELVIN_TO_CELSIUS

REGION = "red_sea"   # options: coral_triangle, andaman_sea, great_barrier_reef, red_sea
YEAR   = 2023
START  = f"{YEAR}-01-01"
END    = f"{YEAR}-12-31"

print(f"Region bounds: {REGIONS[REGION]}")

## 1. Fetch SST

In [ ]:
ds  = fetch.fetch_data(REGION, START, END)
sst = ds["analysed_sst"]
ocean_mask = ~sst.isel(time=0).isnull().compute()
sst

## 2. Climatology (1991–2020 baseline)
90th-percentile threshold per day of year, monthly means, and the maximum
monthly mean (MMM) used as the bleaching baseline.

In [ ]:
clim = detect.load_or_compute_climatology(REGION)
print("Climatology keys:", list(clim.keys()))

In [ ]:
mmm = clim["mmm"].compute() + KELVIN_TO_CELSIUS
fig, ax = plt.subplots(figsize=(10, 6), dpi=100,
                       subplot_kw={"projection": ccrs.PlateCarree()})
ax.add_feature(cfeature.COASTLINE)
ax.add_feature(cfeature.LAND)
mesh = ax.pcolormesh(mmm.longitude, mmm.latitude, mmm.values,
                     transform=ccrs.PlateCarree(), cmap="RdYlBu_r")
cbar = plt.colorbar(mesh, ax=ax, pad=0.02)
cbar.set_label("SST (°C)", fontsize=12)
ax.set_title(f"Maximum Monthly Mean SST — {REGION}")
plt.show()

## 3. Marine heatwaves
Days above the 90th-percentile threshold for at least 5 consecutive days
(Hobday et al. 2016).

In [ ]:
mhw = detect.load_or_compute_mhw(sst, REGION, START, END)
mhw_days = mhw["mhw"].sum(dim="time").compute()
mhw_summary = analyse.summarise_mhw(mhw_days, ocean_mask)
pd.Series(mhw_summary)

In [ ]:
fig = visualise.plot_mhw_days(
    mhw_days,
    hotspot_lat=mhw_summary["hotspot_lat"],
    hotspot_lon=mhw_summary["hotspot_lon"],
    title=f"MHW days {YEAR} — {REGION}",
)
plt.show()

## 4. Degree Heating Weeks
Accumulated hot spots (SST above MMM + 1 °C) over a 12-week window, NOAA CRW method.

In [ ]:
dhw = detect.load_or_compute_dhw(sst, REGION, START, END)
dhw_summary = analyse.summarise_dhw(dhw, ocean_mask)
pd.Series(dhw_summary)

In [ ]:
fig = visualise.plot_dhw(
    dhw,
    hotspot_lat=dhw_summary["hotspot_lat"],
    hotspot_lon=dhw_summary["hotspot_lon"],
    max_dhw=dhw_summary["max_dhw"],
    title=f"Max DHW {YEAR} — {REGION}",
)
plt.show()

## 5. Monthly SST anomaly

In [ ]:
anomaly = analyse.monthly_sst_anomaly(sst, clim["monthly_mean"], ocean_mask)
anomaly

In [ ]:
fig = visualise.plot_sst_anomaly_max(sst, clim, title=f"Max monthly SST anomaly {YEAR} — {REGION}")
plt.show()

## 6. Monthly DHW

In [ ]:
fig = visualise.plot_monthly_matrix(dhw, label="DHW (°C-weeks)", title=f"DHW by month — {REGION} {YEAR}")
plt.show()